# Tahap 5 — Colab satu sel
Jalankan satu sel di bawah. Jika belum ada di panel Files, pilih tiga dataset ketika tombol upload muncul. Semua kode dan protokol sudah tertanam. Tidak perlu Drive, file Python lain, GPU, atau konfigurasi manual. ZIP hasil diunduh pada akhir proses.

Benchmark sintetis; G01 operasional HOLD. Ini revisi kemudahan menjalankan Tahap 5, bukan Tahap 6.


In [ ]:
# TAHAP 5 — SATU SEL COLAB, CUKUP TIGA DATASET
# Jalankan sel ini; pilih tiga file saat tombol upload muncul.
# Tidak memerlukan train_stage5.py, protocol.json, Drive, GPU atau API.
import sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import uuid, json, io, contextlib, shutil

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("Buka notebook ini di Google Colab, lalu jalankan selnya.") from exc

print("[1/5] Memeriksa library. Paket yang sudah ada tidak dipasang ulang.", flush=True)
packages = {"numpy":"numpy", "pandas":"pandas", "sklearn":"scikit-learn>=1.2", "joblib":"joblib", "openpyxl":"openpyxl"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

WORK = Path('/content/AI_Risk_Tahap5') / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
(WORK / 'source').mkdir(parents=True, exist_ok=True)

from pathlib import Path
import argparse, contextlib, hashlib, io, json, math, platform, sys
import numpy as np
import pandas as pd
import sklearn, joblib
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
    precision_score, recall_score, f1_score, confusion_matrix)

BASE = WORK
RAW_NAME = 'project_risk_raw_dataset.csv'
EV_NAME = 'simulated_project_ev_metrics_final.csv.xlsx'
SUMMARY_NAME = 'project_delay_cost_overrun_summary_final.csv.xlsx'

def dump(path, obj):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

def read_table(path):
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path, keep_default_na=False)
    frame = pd.read_excel(path, header=None, keep_default_na=False)
    if frame.shape[1] == 1 and 'Project_ID,' in str(frame.iloc[0, 0]):
        return pd.read_csv(io.StringIO('\n'.join(frame.iloc[:, 0].astype(str))), keep_default_na=False)
    return pd.read_excel(path, keep_default_na=False)

def ranked_indices(ids, score):
    return np.lexsort((np.asarray(ids, dtype=str), -np.asarray(score)))

def rank_metrics(ids, y, score, fraction):
    y = np.asarray(y)
    k = int(math.ceil(len(y) * fraction))
    chosen = ranked_indices(ids, score)[:k]
    hits = int(y[chosen].sum())
    return {'capacity_fraction': fraction, 'n': len(y), 'late': int(y.sum()),
            'k': k, 'hits': hits, 'precision_at_k': hits/k,
            'recall_at_k': hits/int(y.sum()), 'false_alerts_at_k': k-hits}

def metrics(y, score, pred, probability=True):
    return {'n':len(y), 'late':int(np.sum(y)), 'alerts':int(np.sum(pred)),
            'precision':float(precision_score(y,pred,zero_division=0)),
            'recall':float(recall_score(y,pred,zero_division=0)),
            'f1':float(f1_score(y,pred,zero_division=0)),
            'average_precision':float(average_precision_score(y,score)),
            'roc_auc':float(roc_auc_score(y,score)),
            'brier':float(brier_score_loss(y,score)) if probability else None,
            'confusion_matrix_TN_FP_FN_TP':confusion_matrix(y,pred,labels=[0,1]).ravel().tolist()}

def audit_sources(raw, ev, summary):
    assert raw.Project_ID.is_unique and summary.Project_ID.is_unique, 'Duplicate project IDs'
    assert set(raw.Project_ID)==set(ev.Project_ID)==set(summary.Project_ID), 'ID sets differ'
    assert not ev.duplicated(['Project_ID','Month']).any(), 'Duplicate monthly key'
    assert (summary.Schedule_Delay == summary.Actual_Duration_Months-summary.Estimated_Timeline_Months).all()
    assert np.allclose(summary.Cost_Overrun,summary.Actual_Cost-summary.Planned_Budget)
    j=raw[['Project_ID','Project_Budget_USD','Estimated_Timeline_Months']].merge(summary,on='Project_ID',suffixes=('_raw','_summary'),validate='one_to_one')
    assert np.allclose(j.Project_Budget_USD,j.Planned_Budget)
    assert (j.Estimated_Timeline_Months_raw==j.Estimated_Timeline_Months_summary).all()
    e=ev.merge(summary,on='Project_ID',suffixes=('_ev','_summary'),validate='many_to_one')
    for c in ['Estimated_Timeline_Months','Actual_Duration_Months','Planned_Budget','Actual_Cost']:
        assert np.allclose(e[c+'_ev'], e[c+'_summary']), c+' inconsistent'
    assert (ev[['PV','EV','AC','Planned_Budget']]>0).all().all()
    assert all(g.Month.tolist()==list(range(1,len(g)+1)) for _,g in ev.sort_values(['Project_ID','Month']).groupby('Project_ID'))
    at_end=ev[ev.Month.eq(ev.Estimated_Timeline_Months)].merge(summary[['Project_ID','Schedule_Delay']],on='Project_ID')
    late=at_end[at_end.Schedule_Delay.gt(0)]
    delta=ev.sort_values(['Project_ID','Month']).copy()
    for col in ['PV','EV','AC']:
        delta['d'+col]=delta.groupby('Project_ID')[col].diff().fillna(delta[col])
    return {'rows':{'raw':len(raw),'monthly':len(ev),'summary':len(summary)},
      'unique_project_and_monthly_keys':True, 'id_sets_match':True, 'label_duration_consistent':True,
      'cost_budget_duration_reconciliation':True,
      'late_projects_at_planned_end':len(late),
      'late_PV_below_budget_at_planned_end':int((late.PV<late.Planned_Budget-1e-6).sum()),
      'cpi_cumulative_matches_EV_AC':bool(np.allclose(ev.CPI_Cumulative,ev.EV/ev.AC)),
      'spi_cumulative_matches_EV_PV':bool(np.allclose(ev.SPI_Cumulative,ev.EV/ev.PV)),
      'monthly_SPI_not_raw_increment_ratio':int((~np.isclose(delta.SPI_Monthly,delta.dEV/delta.dPV)).sum()),
      'monthly_SPI_matches_clipped_increment_ratio':bool(np.allclose(delta.SPI_Monthly,(delta.dEV/delta.dPV).clip(.6,1.4))),
      'literal_None_preserved_not_missing':{c:int(raw[c].eq('None').sum()) for c in ['Tech_Environment_Stability','Change_Control_Maturity','Risk_Management_Maturity']},
      'literal_NA_unresolved_semantics':{'Tech_Environment_Stability':int(raw.Tech_Environment_Stability.eq('N/A').sum()),'note':'N/A may mean not applicable or unavailable; preserved literally, not imputed or used as predictor'},
      'empty_cells':{'raw':int(raw.eq('').sum().sum()),'monthly':int(ev.eq('').sum().sum()),'summary':int(summary.eq('').sum().sum())},
      'asof_status':'UNVERIFIED: generator code and baseline freeze history unavailable; monthly features provisional synthetic measurements only',
      'operational_gate':'HOLD'}

def prepare(raw,ev,summary):
    # Exact month slices ONLY. Never group all future observations into predictors.
    m1=ev[ev.Month.eq(1)][['Project_ID','EV','AC']].rename(columns={'EV':'EV1','AC':'AC1'})
    m2=ev[ev.Month.eq(2)][['Project_ID','EV','AC']].rename(columns={'EV':'EV2','AC':'AC2'})
    base=raw.loc[raw.Project_Type.eq('IT'),['Project_ID','Project_Budget_USD','Estimated_Timeline_Months']].copy()
    joined=base.merge(m1,on='Project_ID',validate='one_to_one').merge(m2,on='Project_ID',validate='one_to_one')
    eligible=joined.Estimated_Timeline_Months.gt(2)&joined.EV2.lt(joined.Project_Budget_USD)
    d=joined[eligible].copy().sort_values('Project_ID').reset_index(drop=True)
    b=d.Project_Budget_USD
    d['planned_months']=d.Estimated_Timeline_Months
    d['log_budget']=np.log1p(b)
    d['earned_fraction_m2']=d.EV2/b
    d['spent_fraction_m2']=d.AC2/b
    d['cpi_m2']=d.EV2/d.AC2
    d['earned_delta_fraction']=(d.EV2-d.EV1)/b
    d['spent_delta_fraction']=(d.AC2-d.AC1)/b
    # Outcome joined only AFTER feature computation; never selected by predictor list.
    labels=summary[['Project_ID','Schedule_Delay']].assign(delayed=lambda x:x.Schedule_Delay.gt(0).astype(int))
    d=d.merge(labels[['Project_ID','delayed']],on='Project_ID',validate='one_to_one')
    return d, {'IT_before_eligibility':len(base),'after_eligibility':len(d),'excluded':len(base)-len(d),'late':int(d.delayed.sum())}

def run(source_dir=None, output_dir=None, executor='Python runtime; operator identity not inferred'):
    source=Path(source_dir) if source_dir else BASE/'source'
    output=Path(output_dir) if output_dir else BASE
    for sub in ['data','results','model']: (output/sub).mkdir(parents=True,exist_ok=True)
    protocol=json.loads((BASE/'protocol.json').read_text(encoding='utf-8'))
    seed=protocol['seed']; features=protocol['features']
    raw,ev,summary=[TABLE_CACHE[n] for n in [RAW_NAME,EV_NAME,SUMMARY_NAME]]
    audit=audit_sources(raw,ev,summary)
    d,cohort=prepare(raw,ev,summary); audit['cohort']=cohort
    assert d.Project_ID.is_unique and np.isfinite(d[features].to_numpy()).all()
    trainval,test=train_test_split(d,test_size=.2,stratify=d.delayed,random_state=seed)
    train,val=train_test_split(trainval,test_size=.25,stratify=trainval.delayed,random_state=seed)
    frames={'train':train,'validation':val,'test':test}
    ids=[set(x.Project_ID) for x in frames.values()]
    assert not(ids[0]&ids[1] or ids[0]&ids[2] or ids[1]&ids[2])
    split=pd.concat([x[['Project_ID','delayed']].assign(split=k) for k,x in frames.items()]).sort_values('Project_ID')
    split.to_csv(output/'data/split_manifest.csv',index=False)
    d[['Project_ID']+features+['delayed']].merge(split[['Project_ID','split']],on='Project_ID',validate='one_to_one').to_csv(output/'data/snapshot_month2.csv',index=False)
    dump(output/'results/audit.json',audit)
    fitted={}; validation=[]
    for name in protocol['candidates']:
        if name.startswith('LR'):
            c=float(name.split('_C')[1])
            estimator=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),
                ('model',LogisticRegression(C=c,max_iter=3000,random_state=seed))])
        else:
            depth=int(name.split('depth')[1])
            estimator=Pipeline([('impute',SimpleImputer(strategy='median')),
                ('model',RandomForestClassifier(n_estimators=200,max_depth=depth,min_samples_leaf=10,n_jobs=1,random_state=seed))])
        calibrated=CalibratedClassifierCV(estimator,method='sigmoid',cv=StratifiedKFold(3,shuffle=True,random_state=seed),n_jobs=1)
        calibrated.fit(train[features],train.delayed)
        p=calibrated.predict_proba(val[features])[:,1]
        validation.append({'method':name,**metrics(val.delayed,p,p>=.5)})
        fitted[name]=calibrated
    # Stable sort preserves LR/smaller-hyperparameter preference for exact AP ties.
    best=sorted(validation,key=lambda x:-x['average_precision'])[0]['method']
    model=fitted[best]
    rule_val=-val.cpi_m2.to_numpy()
    validation.append({'method':'CPI_rule',**metrics(val.delayed,rule_val,val.cpi_m2.lt(1),False)})
    pd.DataFrame(validation).drop(columns=['confusion_matrix_TN_FP_FN_TP']).to_csv(output/'results/validation_metrics.csv',index=False)
    # Selection locked before test is scored. No train+validation refit.
    selection={'selected':best,'criterion':'validation AP','split_counts':{k:len(x) for k,x in frames.items()},
      'validation_model_AP':next(x['average_precision'] for x in validation if x['method']==best),
      'validation_rule_AP':validation[-1]['average_precision'], 'operational_gate':'HOLD'}
    dump(output/'results/selection.json',selection)
    p=model.predict_proba(test[features])[:,1]; rule=-test.cpi_m2.to_numpy()
    prevalence=float(train.delayed.mean())
    evaluated=[{'method':best,**metrics(test.delayed,p,p>=.5)},
      {'method':'CPI_rule',**metrics(test.delayed,rule,test.cpi_m2.lt(1),False)},
      {'method':'Train_prevalence',**metrics(test.delayed,np.full(len(test),prevalence),np.full(len(test),prevalence>=.5))}]
    dump(output/'results/test_metrics.json',evaluated)
    pd.DataFrame(evaluated).drop(columns=['confusion_matrix_TN_FP_FN_TP']).to_csv(output/'results/test_metrics.csv',index=False)
    rankings=[]
    for fraction in [.1,.2,.3]:
        for name,score in [(best,p),('CPI_rule',rule)]:
            rankings.append({'method':name,**rank_metrics(test.Project_ID,test.delayed,score,fraction)})
    pd.DataFrame(rankings).to_csv(output/'results/capacity_scenarios.csv',index=False)
    predictions=test[['Project_ID','delayed','cpi_m2']].copy()
    predictions['model_score']=p; predictions['rule_score']=rule
    predictions['scope']='synthetic_test_only_not_new_operational_batch'
    predictions.sort_values('Project_ID').to_csv(output/'results/test_predictions.csv',index=False)
    frac_pos,mean_pred=calibration_curve(test.delayed,p,n_bins=5,strategy='quantile')
    pd.DataFrame({'mean_score':mean_pred,'observed_late_fraction':frac_pos}).to_csv(output/'results/calibration_bins.csv',index=False)
    rng=np.random.default_rng(seed); y=test.delayed.to_numpy(); arrids=test.Project_ID.to_numpy()
    diffs=[]
    for _ in range(2000):
        ix=rng.integers(0,len(y),len(y))
        if y[ix].sum()==0: continue
        # Duplicates are normal project bootstrap resampling. Same sample for both methods.
        a=rank_metrics(arrids[ix],y[ix],p[ix],.2)['recall_at_k']
        b=rank_metrics(arrids[ix],y[ix],rule[ix],.2)['recall_at_k']
        diffs.append(a-b)
    uncertainty={'paired_bootstrap_n':len(diffs),'recall_at_20pct_difference_interval95':np.quantile(diffs,[.025,.975]).tolist(),
      'note':'Internal synthetic test only. CI does not establish real-world or causal validity; ties use ID.'}
    dump(output/'results/uncertainty.json',uncertainty)
    # Persist a complete fitted preprocessing + calibration pipeline, not scores only.
    joblib.dump(model,output/'model/exploratory_model.joblib')
    loaded=joblib.load(output/'model/exploratory_model.joblib')
    assert np.allclose(p,loaded.predict_proba(test[features])[:,1])
    # Meaningful leakage regression: edits after month 2 must not change snapshot features.
    changed=ev.copy(); changed.loc[changed.Month.gt(2),['EV','AC']]*=1.5
    changed['Actual_Cost']=-999; changed['Actual_Duration_Months']=-999; changed['PV']=-999
    d2,_=prepare(raw,changed,summary)
    assert np.allclose(d[features],d2[features]) and d.Project_ID.equals(d2.Project_ID)
    assert list(model.feature_names_in_)==features
    checks={'source_reconciliation':True,'project_disjoint':True,'single_snapshot_per_project':True,
      'no_explicit_outcome_PV_SPI_or_ID_predictors':True,'future_row_and_forbidden_column_mutation_invariance':True,
      'reload_prediction_match':True,'feature_values_finite':True,'test_tuning':False,
      'semantic_asof_proven':False,'operational_gate_passed':False,
      'executor':executor}
    dump(output/'results/checks.json',checks)
    environment={'python':platform.python_version(),'platform':platform.platform(),
      'packages':{'numpy':np.__version__,'pandas':pd.__version__,'scikit-learn':sklearn.__version__,'joblib':joblib.__version__},
      'source_hashes':{n:hashlib.sha256((source/n).read_bytes()).hexdigest() for n in [RAW_NAME,EV_NAME,SUMMARY_NAME]},
      'protocol_sha256':hashlib.sha256((BASE/'protocol.json').read_bytes()).hexdigest(),
      'implementation_version':'colab-self-contained-v2'}
    dump(output/'results/environment.json',environment)
    dump(output/'model/model_card.json',{'selected':best,'features':features,'snapshot_month':2,
      'population':'IT synthetic projects eligible at month 2','operational_status':'HOLD',
      'limitations':['Synthetic rule-based generator; no independent real-world test',
      'As-of and baseline frozen timestamps not verified','Only month 2, not all project ages',
      'USD units from source; not IDR PMO build budget','Scores not approved operational probabilities'],
      'train_only_fit':True,'seed':seed})
    print(json.dumps({'selected':best,'cohort':cohort,'splits':selection['split_counts'],'test':evaluated,
      'capacity':rankings,'uncertainty':uncertainty,'checks':checks},indent=2))
    return {'audit':audit,'selection':selection,'metrics':evaluated,'capacity':rankings,'uncertainty':uncertainty}


PROTOCOL_JSON = '{"version": "1.0", "scope": "Exploratory synthetic benchmark only; operational use blocked pending as-of validation", "actual_authoring_date": "2026-11-04", "population": "Project_Type == IT; one observation per project at end of month 2; planned duration > 2 and EV2 < budget", "label": "Schedule_Delay > 0; verify equals Actual_Duration_Months > Estimated_Timeline_Months", "snapshot_month": 2, "seed": 20261104, "split": "60% train, 20% validation, 20% test; stratified by label; project-disjoint", "features": ["planned_months", "log_budget", "earned_fraction_m2", "spent_fraction_m2", "cpi_m2", "earned_delta_fraction", "spent_delta_fraction"], "excluded": "All static risk descriptors with unknown update time; PV/SPI/SV; final cost/duration/delay/overrun; source Risk_Level; project ID", "rule": "Rank lower observed CPI2 first; threshold alert CPI2 < 1. Rule scores are not probabilities.", "candidates": ["LR_C0.1", "LR_C1", "RF_depth4", "RF_depth8"], "calibration": "sigmoid; 3-fold stratified cross-validation inside train only", "selection": "Highest validation average precision among four ML candidates; ties prefer LR then smaller hyperparameter. Compare chosen model against rule; do not force ML adoption.", "threshold": "0.5 fixed for diagnostic precision/recall/F1; capacity scenario 20% fixed illustrative primary; 10% and 30% secondary only", "ranking_ties": "Project_ID ascending; never use outcome to break ties", "uncertainty": "2000 paired project bootstrap resamples for selected model minus rule recall at 20% capacity; fixed seed; descriptive, not proof of external validity", "test_policy": "No test tuning or refit after seeing test. Training artifact remains fitted on train subset only.", "gate": "G01 HOLD: month 1/2 EV/AC and planned budget treated as provisional simulated as-of measurements; no baseline-freeze or generator code proof. Not leakage-free certified, not approved probabilities for real projects.", "operational_capacity": "Not confirmed by owner. Capacity percentages are sensitivity scenarios, not staffing commitments.", "legacy": "No reuse of embedded dashboard scores or old model. New split is internal holdout, not independently collected external evidence. Earlier data inspection is disclosed.", "implementation": "Colab self-contained v2; unchanged features/split/candidates/selection", "environment": "Uses installed compatible packages; installs only missing packages. Versions recorded; cross-version exact equality not guaranteed."}'

# Protokol tertanam: tidak ada file konfigurasi yang perlu Anda unggah.
PROTOCOL = json.loads(PROTOCOL_JSON)
dump(WORK / 'protocol.json', PROTOCOL)
CANONICAL = [RAW_NAME, EV_NAME, SUMMARY_NAME]
SCHEMAS = {
    RAW_NAME: {'Project_ID','Project_Type','Project_Budget_USD','Team_Size'},
    EV_NAME: {'Project_ID','Month','PV','EV','AC','CPI_Cumulative'},
    SUMMARY_NAME: {'Project_ID','Schedule_Delay','Cost_Overrun','Actual_Duration_Months'},
}
TABLE_CACHE = {}
MAPPING = []

def accept_file(original_name, content):
    # Read once, classify from columns, then preserve original bytes under canonical names.
    suffix = Path(original_name).suffix.lower()
    if suffix not in ('.csv','.xlsx'):
        print('Dilewati (bukan CSV/XLSX):', original_name)
        return
    temporary = WORK / ('input_' + uuid.uuid4().hex[:8] + suffix)
    temporary.write_bytes(content)
    try:
        frame = read_table(temporary)
    except Exception as exc:
        raise ValueError(f'File {original_name} tidak dapat dibaca sebagai CSV/Excel: {exc}') from exc
    finally:
        temporary.unlink(missing_ok=True)
    matches = [name for name, required in SCHEMAS.items() if required.issubset(frame.columns)]
    if len(matches) != 1:
        print('Tidak dikenali sebagai salah satu dari tiga dataset:', original_name)
        return
    canonical = matches[0]
    if canonical in TABLE_CACHE:
        print('Duplikat jenis dataset dilewati:', original_name)
        return
    (WORK / 'source' / canonical).write_bytes(content)
    TABLE_CACHE[canonical] = frame
    MAPPING.append({'uploaded_name':original_name,'stored_name':canonical,'rows':len(frame)})
    print('Siap:', original_name, '→', canonical, f'({len(frame):,} baris)', flush=True)

print('[2/5] Menyiapkan tiga dataset.', flush=True)
# Jika ketiganya sudah diunggah melalui panel Files, langsung gunakan lokasi /content.
# Tidak ada pencarian rekursif dan tidak ada akses Google Drive.
for name in CANONICAL:
    direct = Path('/content') / name
    if direct.is_file():
        accept_file(name, direct.read_bytes())

while len(TABLE_CACHE) < 3:
    needed = [name for name in CANONICAL if name not in TABLE_CACHE]
    print('Pilih file berikut (boleh pilih semuanya sekaligus):', *needed, sep='\n- ')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Upload dibatalkan. Jalankan sel kembali dan pilih tiga dataset.')
    before = len(TABLE_CACHE)
    for name, content in uploaded.items():
        accept_file(name, content)
    if len(TABLE_CACHE) == before:
        raise ValueError('Tidak ada dataset baru yang dikenali. Pilih tiga CSV/XLSX sumber proyek, bukan notebook atau ZIP.')

dump(WORK / 'input_mapping.json', MAPPING)
print('[3/5] Audit, snapshot bulan 2, training dan evaluasi sedang berjalan...', flush=True)
log = io.StringIO()
try:
    with contextlib.redirect_stdout(log):
        result = run(source_dir=WORK/'source', output_dir=WORK,
                     executor='Executed in a Google Colab runtime; operator identity not inferred')
finally:
    (WORK / 'execution_log.txt').write_text(log.getvalue(), encoding='utf-8')

print('[4/5] Menyusun laporan dan paket hasil.', flush=True)
selected = result['selection']['selected']
capacity = pd.DataFrame(result['capacity'])
primary = capacity[capacity.capacity_fraction.eq(.2)]
best20 = primary[primary.method.eq(selected)].iloc[0]
rule20 = primary[primary.method.eq('CPI_rule')].iloc[0]
ci = result['uncertainty']['recall_at_20pct_difference_interval95']
test_metrics = result['metrics'][0]
summary_text = (
    '# Hasil Tahap 5 — benchmark sintetis\n\n'
    f'Model pilihan validation: **{selected}**.\n\n'
    f"Cohort: {result['audit']['cohort']['after_eligibility']} proyek IT pada akhir bulan 2. "
    f"Split: {result['selection']['split_counts']}.\n\n"
    f"Test AP {test_metrics['average_precision']:.4f}; ROC-AUC {test_metrics['roc_auc']:.4f}; Brier {test_metrics['brier']:.4f}.\n\n"
    f"Pada kapasitas {int(best20['k'])} review: model menangkap {int(best20['hits'])} kasus terlambat, "
    f"aturan CPI {int(rule20['hits'])}. Interval bootstrap selisih recall: {ci[0]*100:.2f} sampai {ci[1]*100:.2f} poin persentase.\n\n"
    + ('Interval mencakup nol: keunggulan belum konklusif.\n\n' if ci[0] <= 0 <= ci[1] else
       'Interval tidak mencakup nol pada benchmark internal; generalisasi nyata tetap belum terbukti.\n\n')
)
# Always include full machine-readable metrics, even if statistical outcome differs.
summary_text += (
    '**G01 operasional: HOLD.** As-of EV/AC dan frozen baseline belum terbukti. '
    'Data bersifat sintetis berbasis aturan; skor bukan probabilitas tervalidasi perusahaan nyata. '
    'Kapasitas 10/20/30% adalah skenario; bukan kapasitas tim yang sudah disetujui.\n\n'
    'Training dilakukan pada runtime ini. Identitas operator tidak ditebak. '
    'Versi library dicatat di results/environment.json; versi berbeda dapat menghasilkan angka berbeda.\n\n'
    'Model: model/exploratory_model.joblib. Input/fitur: data/snapshot_month2.csv. '
    'Split: data/split_manifest.csv. Evaluasi lengkap: results/. '
    'Tidak menjalankan Tahap 6, integrasi batch proyek baru atau Power BI.\n\n'
    'Sumber: Reyhane Hajihoseyni (2026), Mendeley Data v1, DOI 10.17632/2p5sz57wh2.1, CC BY 4.0. '
    'https://data.mendeley.com/datasets/2p5sz57wh2/1 . '
    'Perubahan: decoding CSV dalam XLSX, rekonstruksi snapshot, fitur, split dan eksperimen; sumber asli tidak diedit.\n'
)
(WORK / 'LAPORAN_HASIL.md').write_text(summary_text, encoding='utf-8')
pd.DataFrame([{'feature':name,'scope':'synthetic month-2; as-of unverified'} for name in PROTOCOL['features']]).to_csv(WORK/'data/feature_list.csv',index=False)
archive = Path(shutil.make_archive(str(WORK.parent / (WORK.name + '_HASIL')), 'zip', WORK))
print('\nSELESAI — model:', selected)
print(capacity.to_string(index=False))
print('\nGate operasional: HOLD. Baca LAPORAN_HASIL.md dan hasil evaluasi.')
print('Paket hasil:', archive)
print('[5/5] Memicu unduhan ZIP hasil...')
try:
    files.download(str(archive))
except Exception as exc:
    print('Unduhan otomatis tidak berhasil dipicu:', exc)
    print('ZIP tetap tersedia di panel Files Colab:', archive)
